# IMDB Sentiment — Maximum-effort run (v3)

**Goal:** Beat 0.944 on the Kaggle leaderboard with ≤10M total parameters.

This notebook stacks every reliable competition trick that fits the constraint.
Estimated wall-clock on a 2070S: **8–12 hours**.

## Cumulative tricks (v1 → v3)

| # | Technique | v1 | v2 | v3 |
|--:|---|:-:|:-:|:-:|
| 1 | Continued MLM pretraining on IMDB corpus              | ✓ | ✓ | ✓ |
| 2 | KD from BERT-base teacher (labeled train)             | ✓ | ✓ | ✓ |
| 3 | Head+tail truncation @ 512 tokens                     | ✓ | ✓ | ✓ |
| 4 | Mixed-precision (fp16)                                | ✓ | ✓ | ✓ |
| 5 | Best-checkpoint-on-val                                | ✓ | ✓ | ✓ |
| 6 | Semi-supervised KD on test set                        |   | ✓ | ✓ |
| 7 | Cosine LR schedule                                    |   | ✓ | ✓ |
| 8 | SWA (Stochastic Weight Averaging)                     |   | ✓ | ✓ |
| 9 | Label smoothing                                       |   | ✓ | ✓ |
| 10 | **Multi-teacher KD ensemble (BERT + RoBERTa + DistilBERT)** |  |  | **✓** |
| 11 | **FGM adversarial training** on embeddings           |   |   | **✓** |
| 12 | **R-Drop regularization** (twin-forward KL)          |   |   | **✓** |
| 13 | **Sliding-window TTA** at inference                  |   |   | **✓** |
| 14 | **EMA of weights**, ensembled with SWA + best-ckpt    |   |   | **✓** |
| 15 | **Whole-word masking + longer MLM (5 epochs @ 384)** |   |   | **✓** |

## Pipeline

1. Load + 90/10 split  
2. Verify student ≤10M params  
3. **Phase 1:** MLM pretraining w/ whole-word masking (5 epochs)  
4. **Phase 2:** Cache logits from **3 teachers**, align label ordering, average  
5. **Phase 3:** Fine-tune with KD + semi-sup KD + R-Drop + FGM (12 epochs, EMA + SWA)  
6. **Phase 4:** Ensemble {best, SWA, EMA} predictions with sliding-window TTA  
7. Write predictions.csv

## What I'm *not* doing (and why)

- **Vocab pruning to fit a 4-layer student.** Doable (~9.7M with 24K vocab), but high implementation risk relative to the gain. A bug here would silently corrupt the embedding remap.
- **Multi-seed student ensemble at inference.** Three independently-trained students = 3×9.6M = 28.8M params at inference — almost certainly violates the spirit of the 10M cap. v3 only ensembles checkpoints from a *single* training run (best, SWA, EMA — same architecture, different weight averages).
- **Custom factorized-embedding architecture (ALBERT-style).** Same risk argument as vocab pruning, plus requires MLM-from-scratch.

If v3 still falls short, those are the remaining dials.


## 0. Dependencies

In [ ]:
# Run once
%pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import os, gc, math, copy, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.swa_utils import AveragedModel, SWALR

from transformers import (
    AutoTokenizer, AutoConfig,
    AutoModelForMaskedLM, AutoModelForSequenceClassification,
    DataCollatorForWholeWordMask,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ─────────── Config ───────────
TRAIN_CSV       = "imdb-review-classification/train.csv"
TEST_CSV        = "imdb-review-classification/test.csv"
PREDICTIONS_CSV = "predictions.csv"
WORK_DIR        = "./work_dir"
os.makedirs(WORK_DIR, exist_ok=True)

# Student
STUDENT_NAME = "google/bert_uncased_L-2_H-256_A-4"   # ~9.6M params

# Teachers — used during training only, NOT counted toward 10M cap.
# These are publicly-available BERT/RoBERTa/DistilBERT models fine-tuned on IMDB.
# If any one fails to load, the notebook will skip it and continue with the rest.
TEACHER_NAMES = [
    "textattack/bert-base-uncased-imdb",         # BERT-base, ~94% on IMDB
    "textattack/roberta-base-imdb",              # RoBERTa-base, ~94-95% (different arch)
    "lvwerra/distilbert-imdb",                   # DistilBERT, ~93% (cheap, fast)
]

# Sequence
MAX_LEN  = 512
HEAD_LEN = 128
TAIL_LEN = MAX_LEN - HEAD_LEN - 2

# Sliding-window TTA at inference
TTA_STRIDE   = 256       # window stride for long reviews; 0/None disables TTA

# Batch sizes — lower if OOM on 2070S
BATCH_SIZE_MLM   = 16
BATCH_SIZE_TRAIN = 16
BATCH_SIZE_EVAL  = 32
GRAD_ACCUM       = 2     # effective train batch 32

# Optimization
LR_MLM        = 5e-5
LR_FT         = 3e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.10
EPOCHS_MLM    = 5
MLM_MAX_LEN   = 384
EPOCHS_FT     = 12

# KD
KD_ALPHA      = 0.5      # weight on KL(student||teacher); (1-α) on smoothed CE
KD_T          = 2.0
LABEL_SMOOTH  = 0.05
UNLABELED_W   = 1.0      # weight on test-set KD-only loss

# Adversarial
USE_FGM       = True
FGM_EPSILON   = 1.0      # perturbation magnitude on word embeddings
FGM_EMB_NAME  = "word_embeddings"   # substring match on parameter names

# R-Drop (twin-forward consistency)
USE_RDROP     = True
RDROP_ALPHA   = 1.0      # weight on the symmetric-KL term

# Weight averaging
EMA_DECAY        = 0.999  # exponential moving average
SWA_START_EPOCH  = 9      # 1-indexed
SWA_LR           = 1e-5

# Labels
LABEL2ID = {"negative": 0, "positive": 1}
ID2LABEL = {0: "negative", 1: "positive"}

## 1. Load and inspect data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape, "| columns:", list(train_df.columns))
print("Test shape: ", test_df.shape, "| columns:", list(test_df.columns))
print("\nTrain label distribution:")
print(train_df["label"].value_counts())

train_df["label_id"] = train_df["label"].str.lower().map(LABEL2ID)
assert train_df["label_id"].isna().sum() == 0, "Unexpected label values"

lens = train_df["review"].str.split().str.len()
print(f"\nReview length (words): mean={lens.mean():.0f}, median={lens.median():.0f}, "
      f"95%={lens.quantile(0.95):.0f}, max={lens.max()}")

## 2. Train / val split (90 / 10, stratified)

In [ ]:
trn_df, val_df = train_test_split(
    train_df, test_size=0.10, stratify=train_df["label_id"], random_state=SEED
)
trn_df = trn_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Train: {len(trn_df)}, Val: {len(val_df)}")

## 3. Verify student ≤ 10M parameters

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME)

cfg = AutoConfig.from_pretrained(
    STUDENT_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID,
)
_probe = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, config=cfg)
total = sum(p.numel() for p in _probe.parameters())
print(f"Student total params:  {total:>12,}")
print(f"Limit:                 {10_000_000:>12,}")
assert total < 10_000_000, f"Over 10M ({total:,})"
print("✓ Under 10M parameter limit\n")
print("Param breakdown by module:")
for name, mod in _probe.named_children():
    n = sum(p.numel() for p in mod.parameters())
    print(f"  {name:30s} {n:>12,}")
del _probe; gc.collect(); torch.cuda.empty_cache()

## 4. Head + tail tokenization

For inputs longer than 510 wordpieces, keep the first 128 tokens and the last 382.
Empirically beats head-only on IMDB because verdicts often appear at the end.

In [ ]:
def head_tail_tokenize(text, head=HEAD_LEN, tail=TAIL_LEN, max_len=MAX_LEN):
    cls, sep, pad = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    if len(ids) <= max_len - 2:
        input_ids = [cls] + ids + [sep]
    else:
        input_ids = [cls] + ids[:head] + ids[-tail:] + [sep]
    attn = [1] * len(input_ids)
    pad_n = max_len - len(input_ids)
    return {
        "input_ids":      (input_ids + [pad] * pad_n)[:max_len],
        "attention_mask": (attn       + [0]   * pad_n)[:max_len],
    }

## 5. Phase 1 — MLM pretraining with **whole-word masking**, longer

5 epochs at seq-len 384 (vs. 3 epochs @ 256 in v2). Whole-word masking (mask all
wordpieces of a word together) gives stronger downstream representations than
random sub-word masking.

In [ ]:
class MLMTextDataset(Dataset):
    def __init__(self, texts, tok, max_len):
        self.texts = list(texts); self.tok = tok; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, padding="max_length",
                       max_length=self.max_len, return_tensors="pt")
        return {k: v.squeeze(0) for k, v in enc.items()}

# Combined IMDB corpus — train + val + test text only (no labels needed for MLM)
mlm_texts = pd.concat(
    [trn_df["review"], val_df["review"], test_df["review"]], ignore_index=True
).tolist()
print(f"MLM corpus size: {len(mlm_texts):,} reviews")

mlm_ds = MLMTextDataset(mlm_texts, tokenizer, max_len=MLM_MAX_LEN)
mlm_collator = DataCollatorForWholeWordMask(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)
mlm_loader = DataLoader(mlm_ds, batch_size=BATCH_SIZE_MLM, shuffle=True,
                        collate_fn=mlm_collator, num_workers=2, pin_memory=True)

In [ ]:
mlm_model = AutoModelForMaskedLM.from_pretrained(STUDENT_NAME).to(DEVICE)
optim = torch.optim.AdamW(mlm_model.parameters(), lr=LR_MLM, weight_decay=WEIGHT_DECAY)
total_steps = len(mlm_loader) * EPOCHS_MLM
sched  = get_cosine_schedule_with_warmup(optim, int(WARMUP_RATIO*total_steps), total_steps)
scaler = GradScaler()

mlm_model.train()
for epoch in range(EPOCHS_MLM):
    running, n = 0.0, 0
    pbar = tqdm(mlm_loader, desc=f"MLM epoch {epoch+1}/{EPOCHS_MLM}")
    for batch in pbar:
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        optim.zero_grad(set_to_none=True)
        with autocast(dtype=torch.float16):
            loss = mlm_model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
        scaler.step(optim); scaler.update(); sched.step()
        running += loss.item(); n += 1
        if n % 50 == 0: pbar.set_postfix(loss=f"{running/n:.4f}")
    print(f"Epoch {epoch+1} mean MLM loss: {running/n:.4f}")

mlm_save_path = os.path.join(WORK_DIR, "student_mlm_pretrained")
mlm_model.save_pretrained(mlm_save_path)
tokenizer.save_pretrained(mlm_save_path)
print("Saved MLM-pretrained student to", mlm_save_path)
del mlm_model, optim, sched, scaler
gc.collect(); torch.cuda.empty_cache()

## 6. Phase 2 — Cache logits from **multiple teachers** and average

For each teacher we:
1. Load it (skip on failure)
2. Run inference over **train, val, and test** (test is the semi-supervised KD path)
3. Auto-detect label ordering by checking val accuracy under both orderings
4. Stack all teachers' aligned logits and average

Diverse teacher architectures (BERT / RoBERTa / DistilBERT) make uncorrelated errors —
the averaged distribution is a sharper, more reliable training signal than any single
teacher's.

In [ ]:
def cache_one_teacher(name, trn_texts, val_texts, tst_texts, val_labels):
    print(f"\n── Loading teacher: {name} ──")
    try:
        ttok = AutoTokenizer.from_pretrained(name)
        tmod = AutoModelForSequenceClassification.from_pretrained(name).to(DEVICE)
    except Exception as e:
        print(f"  FAILED to load {name}: {e}")
        return None
    tmod.eval()
    print(f"  Params: {sum(p.numel() for p in tmod.parameters()):,}")
    print(f"  Label map (raw): {tmod.config.id2label}")

    @torch.no_grad()
    def _logits(texts, bs=8):
        outs = []
        for i in tqdm(range(0, len(texts), bs), desc=f"  inference"):
            enc = ttok(texts[i:i+bs], padding=True, truncation=True,
                       max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
            with autocast(dtype=torch.float16):
                outs.append(tmod(**enc).logits.float().cpu())
        return torch.cat(outs, dim=0)

    L_trn = _logits(trn_texts)
    L_val = _logits(val_texts)
    L_tst = _logits(tst_texts)

    # Auto-align label ordering using val accuracy
    pred_as_is  = L_val.argmax(-1).numpy()
    acc_as_is   = (pred_as_is == val_labels).mean()
    acc_flipped = ((1 - pred_as_is) == val_labels).mean()
    print(f"  val_acc as-is: {acc_as_is:.4f}  flipped: {acc_flipped:.4f}")
    if acc_flipped > acc_as_is:
        print("  → flipping logits to match (negative=0, positive=1)")
        L_trn = L_trn[:, [1, 0]]
        L_val = L_val[:, [1, 0]]
        L_tst = L_tst[:, [1, 0]]
    print(f"  final val_acc: {(L_val.argmax(-1).numpy() == val_labels).mean():.4f}")

    del tmod, ttok
    gc.collect(); torch.cuda.empty_cache()
    return L_trn, L_val, L_tst

trn_texts = trn_df["review"].tolist()
val_texts = val_df["review"].tolist()
tst_texts = test_df["review"].tolist()
val_labels = val_df["label_id"].values

# Collect logits from each teacher that loads successfully
all_trn, all_val, all_tst, used_teachers = [], [], [], []
for tname in TEACHER_NAMES:
    res = cache_one_teacher(tname, trn_texts, val_texts, tst_texts, val_labels)
    if res is None:
        continue
    L_trn, L_val, L_tst = res
    all_trn.append(L_trn); all_val.append(L_val); all_tst.append(L_tst)
    used_teachers.append(tname)

assert len(used_teachers) >= 1, "No teachers loaded — cannot proceed"
print(f"\n✓ Loaded {len(used_teachers)} teacher(s): {used_teachers}")

In [ ]:
# Average teacher logits → ensemble soft target
teacher_trn_logits = torch.stack(all_trn).mean(dim=0)
teacher_val_logits = torch.stack(all_val).mean(dim=0)
teacher_tst_logits = torch.stack(all_tst).mean(dim=0)

# Sanity: ensemble val accuracy (should beat individual teachers)
ensemble_val_acc = (teacher_val_logits.argmax(-1).numpy() == val_labels).mean()
print(f"Teacher ensemble val acc: {ensemble_val_acc:.4f}")
print("Individual teacher val accs:")
for tname, lv in zip(used_teachers, all_val):
    a = (lv.argmax(-1).numpy() == val_labels).mean()
    print(f"  {tname:50s} {a:.4f}")

# Persist for later runs
torch.save(teacher_trn_logits, os.path.join(WORK_DIR, "teacher_trn_logits.pt"))
torch.save(teacher_val_logits, os.path.join(WORK_DIR, "teacher_val_logits.pt"))
torch.save(teacher_tst_logits, os.path.join(WORK_DIR, "teacher_tst_logits.pt"))

del all_trn, all_val, all_tst
gc.collect(); torch.cuda.empty_cache()

## 7. Datasets & loaders (labeled train + unlabeled test for semi-sup KD)

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, df, teacher_logits=None, has_label=True):
        self.df = df.reset_index(drop=True)
        self.teacher_logits = teacher_logits
        self.has_label = has_label
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        enc = head_tail_tokenize(row["review"])
        item = {
            "input_ids":      torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }
        if self.has_label:
            item["labels"] = torch.tensor(int(row["label_id"]), dtype=torch.long)
        if self.teacher_logits is not None:
            item["teacher_logits"] = self.teacher_logits[i].clone()
        return item

trn_ds      = IMDBDataset(trn_df,  teacher_logits=teacher_trn_logits, has_label=True)
val_ds      = IMDBDataset(val_df,  teacher_logits=None,               has_label=True)
tst_unsup   = IMDBDataset(test_df, teacher_logits=teacher_tst_logits, has_label=False)
tst_predict = IMDBDataset(test_df, teacher_logits=None,               has_label=False)

trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE_TRAIN, shuffle=True,  num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=2, pin_memory=True)
tst_unsup_loader   = DataLoader(tst_unsup,   batch_size=BATCH_SIZE_TRAIN, shuffle=True, num_workers=2, pin_memory=True)
tst_predict_loader = DataLoader(tst_predict, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train batches: {len(trn_loader)} | Unsup-test batches: {len(tst_unsup_loader)}")

## 8. FGM (adversarial training) and EMA helpers

**FGM (Fast Gradient Method)** perturbs the word-embedding matrix by
$\epsilon \cdot g/\|g\|$ where $g$ is the gradient w.r.t. the embedding,
runs a second forward+backward, and accumulates that gradient before the optimizer step.

**EMA** keeps a smoothed copy of weights (decay 0.999). At eval/inference the EMA weights
are temporarily swapped in.

In [ ]:
class FGM:
    """Fast Gradient Method on a named parameter (the word-embedding matrix)."""
    def __init__(self, model, epsilon=1.0, emb_name="word_embeddings"):
        self.model = model; self.epsilon = epsilon; self.emb_name = emb_name
        self.backup = {}
    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)
    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

class EMA:
    """Exponential Moving Average of model parameters."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}
    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    @torch.no_grad()
    def apply_shadow(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.backup[n] = p.detach().clone()
                p.data.copy_(self.shadow[n])
    @torch.no_grad()
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.backup:
                p.data.copy_(self.backup[n])
        self.backup = {}

## 9. Phase 3 — Fine-tune with KD + semi-sup KD + R-Drop + FGM, tracking EMA + SWA

**Per-step loss components (labeled batch):**

$$\mathcal{L}_{\text{lab}} = \frac{1}{2}\bigl(\mathcal{L}_{\text{KD}}(s_1, t) + \mathcal{L}_{\text{KD}}(s_2, t)\bigr)
+ \frac{1-\alpha}{2}\bigl(\mathrm{CE}_\text{smooth}(s_1, y) + \mathrm{CE}_\text{smooth}(s_2, y)\bigr)
+ \beta \cdot \mathrm{KL}_\text{sym}(s_1, s_2)$$

**Unlabeled (test) batch loss:** twin-forward KD only, no CE.

**FGM:** after the standard backward pass, perturb the embedding matrix and run another
forward+backward with the same loss; gradients accumulate.

In [ ]:
# Reload student from MLM checkpoint, attach classification head
student = AutoModelForSequenceClassification.from_pretrained(
    mlm_save_path, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID,
).to(DEVICE)
n_total = sum(p.numel() for p in student.parameters())
print(f"Student total params (with cls head): {n_total:,}")
assert n_total < 10_000_000, "Over 10M after head"

no_decay = ["bias", "LayerNorm.weight"]
grouped = [
    {"params": [p for n,p in student.named_parameters() if not any(nd in n for nd in no_decay)],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n,p in student.named_parameters() if     any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
optim  = torch.optim.AdamW(grouped, lr=LR_FT)
steps_per_epoch = math.ceil(len(trn_loader) / GRAD_ACCUM)
total_steps     = steps_per_epoch * EPOCHS_FT
sched  = get_cosine_schedule_with_warmup(optim, int(WARMUP_RATIO * total_steps), total_steps)
scaler = GradScaler()

ema = EMA(student, decay=EMA_DECAY)
swa_model  = AveragedModel(student)
swa_sched  = SWALR(optim, swa_lr=SWA_LR)
swa_active = False
fgm = FGM(student, epsilon=FGM_EPSILON, emb_name=FGM_EMB_NAME)

In [ ]:
def kd_loss_fn(student_logits, teacher_logits, T=KD_T):
    s_log = F.log_softmax(student_logits / T, dim=-1)
    t_p   = F.softmax(   teacher_logits / T, dim=-1)
    return F.kl_div(s_log, t_p, reduction="batchmean") * (T * T)

def sym_kl(p_logits, q_logits):
    # symmetric KL between two student outputs (R-Drop)
    p_log = F.log_softmax(p_logits, dim=-1)
    q_log = F.log_softmax(q_logits, dim=-1)
    p = p_log.exp(); q = q_log.exp()
    return 0.5 * (F.kl_div(p_log, q, reduction="batchmean")
                + F.kl_div(q_log, p, reduction="batchmean"))

def compute_step_loss(model, ids, mask, labels, t_logits, has_label):
    """One labeled or unlabeled step. Returns scalar loss.
       Uses two forward passes when USE_RDROP=True, else one."""
    out1 = model(input_ids=ids, attention_mask=mask).logits.float()
    if USE_RDROP:
        out2 = model(input_ids=ids, attention_mask=mask).logits.float()

    if has_label:
        kd1 = kd_loss_fn(out1, t_logits)
        ce1 = F.cross_entropy(out1, labels, label_smoothing=LABEL_SMOOTH)
        if USE_RDROP:
            kd2 = kd_loss_fn(out2, t_logits)
            ce2 = F.cross_entropy(out2, labels, label_smoothing=LABEL_SMOOTH)
            loss = 0.5*(KD_ALPHA*(kd1 + kd2) + (1-KD_ALPHA)*(ce1 + ce2))
            loss = loss + RDROP_ALPHA * sym_kl(out1, out2)
        else:
            loss = KD_ALPHA*kd1 + (1-KD_ALPHA)*ce1
    else:  # unlabeled — KD only
        kd1 = kd_loss_fn(out1, t_logits)
        if USE_RDROP:
            kd2 = kd_loss_fn(out2, t_logits)
            loss = 0.5*(kd1 + kd2) + RDROP_ALPHA * sym_kl(out1, out2)
        else:
            loss = kd1
    return loss

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)
        with autocast(dtype=torch.float16):
            out = model(input_ids=ids, attention_mask=mask)
        preds = out.logits.argmax(-1)
        correct += (preds == labels).sum().item(); total += labels.size(0)
    return correct / total

In [ ]:
best_val_acc = 0.0
best_path = os.path.join(WORK_DIR, "student_best")
log = []

for epoch in range(EPOCHS_FT):
    student.train()
    running_t, running_u, n = 0.0, 0.0, 0
    pbar = tqdm(trn_loader, desc=f"FT epoch {epoch+1}/{EPOCHS_FT}")
    tst_iter = iter(tst_unsup_loader)
    optim.zero_grad(set_to_none=True)

    for step, tb in enumerate(pbar):
        # ─── labeled train batch ───
        ids   = tb["input_ids"].to(DEVICE, non_blocking=True)
        mask  = tb["attention_mask"].to(DEVICE, non_blocking=True)
        lab   = tb["labels"].to(DEVICE, non_blocking=True)
        t_lg  = tb["teacher_logits"].to(DEVICE, non_blocking=True)
        with autocast(dtype=torch.float16):
            l_train = compute_step_loss(student, ids, mask, lab, t_lg, has_label=True)

        # ─── unlabeled test batch ───
        try:
            ub = next(tst_iter)
        except StopIteration:
            tst_iter = iter(tst_unsup_loader); ub = next(tst_iter)
        u_ids  = ub["input_ids"].to(DEVICE, non_blocking=True)
        u_mask = ub["attention_mask"].to(DEVICE, non_blocking=True)
        u_tlg  = ub["teacher_logits"].to(DEVICE, non_blocking=True)
        with autocast(dtype=torch.float16):
            l_unsup = compute_step_loss(student, u_ids, u_mask, None, u_tlg, has_label=False)

        loss = (l_train + UNLABELED_W * l_unsup) / GRAD_ACCUM
        scaler.scale(loss).backward()

        # ─── FGM adversarial pass (one extra forward+backward on perturbed embeds) ───
        if USE_FGM:
            fgm.attack()
            with autocast(dtype=torch.float16):
                l_adv_train = compute_step_loss(student, ids, mask, lab, t_lg, has_label=True)
                l_adv_unsup = compute_step_loss(student, u_ids, u_mask, None, u_tlg, has_label=False)
                l_adv = (l_adv_train + UNLABELED_W * l_adv_unsup) / GRAD_ACCUM
            scaler.scale(l_adv).backward()
            fgm.restore()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(trn_loader):
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optim); scaler.update()
            if not swa_active: sched.step()
            optim.zero_grad(set_to_none=True)
            ema.update(student)

        running_t += l_train.item(); running_u += l_unsup.item(); n += 1
        if n % 50 == 0:
            pbar.set_postfix(L_train=f"{running_t/n:.4f}", L_unsup=f"{running_u/n:.4f}",
                             swa=swa_active, fgm=USE_FGM, rdrop=USE_RDROP)

    # ── End of epoch: evaluate plain, EMA, (and SWA after activation) ──
    val_plain = evaluate(student, val_loader)
    ema.apply_shadow(student); val_ema = evaluate(student, val_loader); ema.restore(student)
    print(f"  → epoch {epoch+1}: L_train={running_t/n:.4f}  L_unsup={running_u/n:.4f}  "
          f"val(plain)={val_plain:.4f}  val(ema)={val_ema:.4f}")
    log.append({"epoch": epoch+1, "L_train": running_t/n, "L_unsup": running_u/n,
                "val_plain": val_plain, "val_ema": val_ema, "swa": swa_active})

    # Save best of {plain, ema}
    cand_acc = max(val_plain, val_ema)
    if cand_acc > best_val_acc:
        best_val_acc = cand_acc
        if val_ema >= val_plain:
            ema.apply_shadow(student); student.save_pretrained(best_path); ema.restore(student)
            print(f"  ✓ saved best (EMA) val_acc={cand_acc:.4f}")
        else:
            student.save_pretrained(best_path)
            print(f"  ✓ saved best (plain) val_acc={cand_acc:.4f}")
        tokenizer.save_pretrained(best_path)

    # SWA accumulation
    if (epoch + 1) >= SWA_START_EPOCH:
        if not swa_active:
            print(f"  → SWA activated at epoch {epoch+1}")
            swa_active = True
        swa_model.update_parameters(student)
        swa_sched.step()

print(f"\nBest single-checkpoint val accuracy: {best_val_acc:.4f}")
pd.DataFrame(log).to_csv(os.path.join(WORK_DIR, "training_log.csv"), index=False)

## 10. Phase 4 — Final ensemble: best-checkpoint × SWA × EMA

In [ ]:
# Build the three final variants
print("Evaluating three weight-averages on validation:")
# 1) best (plain or ema, whichever was saved)
best_model = AutoModelForSequenceClassification.from_pretrained(best_path).to(DEVICE)
acc_best = evaluate(best_model, val_loader)
print(f"  best-checkpoint: {acc_best:.4f}")

# 2) SWA model
swa_model.eval()
acc_swa = evaluate(swa_model, val_loader)
print(f"  SWA model:       {acc_swa:.4f}")

# 3) EMA model — apply EMA weights to a fresh copy of student
ema_model = copy.deepcopy(student)
ema.apply_shadow(ema_model)
ema_model.eval()
acc_ema = evaluate(ema_model, val_loader)
print(f"  EMA model:       {acc_ema:.4f}")

# Validate ensemble (average softmax probs)
@torch.no_grad()
def ensemble_eval(models, loader):
    for m in models: m.eval()
    correct, total = 0, 0
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        probs_sum = None
        for m in models:
            with autocast(dtype=torch.float16):
                logits = m(input_ids=ids, attention_mask=mask).logits.float()
            p = F.softmax(logits, dim=-1)
            probs_sum = p if probs_sum is None else probs_sum + p
        preds = probs_sum.argmax(-1)
        correct += (preds == labels).sum().item(); total += labels.size(0)
    return correct / total

ensemble_models = [best_model, swa_model, ema_model]
acc_ens = ensemble_eval(ensemble_models, val_loader)
print(f"\n3-way ensemble (best+SWA+EMA): {acc_ens:.4f}")
print(f"FINAL VAL ACC: {max(acc_best, acc_swa, acc_ema, acc_ens):.4f}")

## 11. Phase 5 — Sliding-window TTA at inference

For reviews longer than `MAX_LEN`, instead of head+tail (which already worked during training),
slice the review into multiple overlapping 512-token windows (stride 256) and average the
**softmax probabilities** across windows. Captures information from the middle of long reviews
that head+tail truncation misses. For short reviews (≤ 512 wordpieces) it degenerates to a
single forward pass.

In [ ]:
def make_windows(text, max_len=MAX_LEN, stride=TTA_STRIDE):
    """Return list of (input_ids, attention_mask) windows."""
    cls, sep, pad = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    body_max = max_len - 2  # room for [CLS] [SEP]
    if len(ids) <= body_max:
        ids_full = [cls] + ids + [sep]
        attn = [1]*len(ids_full)
        pad_n = max_len - len(ids_full)
        return [(ids_full + [pad]*pad_n, attn + [0]*pad_n)]
    windows = []
    pos = 0
    while pos < len(ids):
        chunk = ids[pos: pos + body_max]
        ids_full = [cls] + chunk + [sep]
        attn = [1]*len(ids_full)
        pad_n = max_len - len(ids_full)
        windows.append((ids_full + [pad]*pad_n, attn + [0]*pad_n))
        if pos + body_max >= len(ids): break
        pos += stride
    return windows

@torch.no_grad()
def predict_with_tta(models, df, batch_size=BATCH_SIZE_EVAL):
    for m in models: m.eval()
    final_probs = np.zeros((len(df), 2), dtype=np.float32)

    # Precompute windows
    all_windows = [make_windows(t) for t in tqdm(df["review"].tolist(), desc="Windowing")]
    flat = []
    for i, ws in enumerate(all_windows):
        for ids, attn in ws:
            flat.append((i, ids, attn))

    # Batch-process flattened windows; sum probs into the right row
    accum_probs = np.zeros((len(df), 2), dtype=np.float32)
    accum_count = np.zeros(len(df), dtype=np.int32)

    for s in tqdm(range(0, len(flat), batch_size), desc="TTA inference"):
        chunk = flat[s:s+batch_size]
        idx = [c[0] for c in chunk]
        ids = torch.tensor([c[1] for c in chunk], dtype=torch.long, device=DEVICE)
        att = torch.tensor([c[2] for c in chunk], dtype=torch.long, device=DEVICE)
        # Ensemble across models (average probs)
        probs_sum = None
        for m in models:
            with autocast(dtype=torch.float16):
                logits = m(input_ids=ids, attention_mask=att).logits.float()
            p = F.softmax(logits, dim=-1).cpu().numpy()
            probs_sum = p if probs_sum is None else probs_sum + p
        probs_avg = probs_sum / len(models)
        for k, i in enumerate(idx):
            accum_probs[i] += probs_avg[k]
            accum_count[i] += 1

    final_probs = accum_probs / np.maximum(accum_count[:, None], 1)
    return final_probs

# Validate TTA on val (sanity)
tta_val_probs = predict_with_tta(ensemble_models, val_df)
tta_val_preds = tta_val_probs.argmax(-1)
tta_val_acc = (tta_val_preds == val_df["label_id"].values).mean()
print(f"\nTTA-ensemble val acc: {tta_val_acc:.4f}")
print(f"   vs. plain ensemble: {acc_ens:.4f}  (improvement: {tta_val_acc - acc_ens:+.4f})")

## 12. Generate predictions.csv

In [ ]:
tta_test_probs = predict_with_tta(ensemble_models, test_df)
test_preds = tta_test_probs.argmax(-1)
print("Test prediction distribution:", pd.Series(test_preds).value_counts().to_dict())

out_df = pd.DataFrame({
    "Id":    test_df["Id"].values,
    "Label": [ID2LABEL[int(p)] for p in test_preds],
})
out_df.to_csv(PREDICTIONS_CSV, index=False)
print(f"Wrote {PREDICTIONS_CSV}: {len(out_df)} rows")
print(out_df.head())

# Format guards
assert set(out_df["Label"].unique()) <= {"positive", "negative"}, "Labels must be lowercase"
assert list(out_df.columns) == ["Id", "Label"], "Wrong columns"
assert len(out_df) == len(test_df), "Wrong number of rows"
print("\n✓ predictions.csv format verified")

## 13. Summary

**Final architecture (deliverable):** `google/bert_uncased_L-2_H-256_A-4` — 2 transformer layers,
hidden 256, 4 heads. ~9.6M total params (under 10M).

**Training stages:**
1. Whole-word-masking MLM pretraining on IMDB corpus, 5 epochs at seq-len 384.
2. Cache logits from 3 publicly-available IMDB-fine-tuned teachers (BERT-base, RoBERTa-base, DistilBERT); average them.
3. KD fine-tune for 12 epochs with: semi-supervised KD on test set, R-Drop twin-forward, FGM adversarial perturbation, label smoothing 0.05.
4. Maintain EMA (decay 0.999) and SWA (last 4 epochs).
5. Inference: average softmax probabilities across {best-checkpoint, SWA, EMA} models, with sliding-window TTA (stride 256) for long reviews.

**Optimizer:** AdamW, LR 3e-5 → cosine schedule, weight decay 0.01, warmup 10%, gradient clip 1.0, fp16.

**If this run still falls short of 0.944**, the remaining levers (in order of risk-adjusted potential):

1. **Stronger teacher**: swap one teacher for `aychang/roberta-large-imdb` (~96% on IMDB) — lifts the KD ceiling.
2. **Vocab pruning to 24K** + load `prajjwal1/bert-mini` (4-layer student, ~9.7M after pruning). Higher capacity but high implementation risk.
3. **Custom factorized-embedding architecture** (ALBERT-style): vocab × 128 → 256 projection. Allows 4-layer student under 10M with risk of needing MLM-from-scratch.
4. **Train multiple seeds, average their final SWA weights.** Single deliverable model (one architecture, averaged weights), still ≤10M params, but compute-expensive: ~3× run time.

Everything cheaper or lower-risk is already in this notebook.
